In [0]:
from pyspark.sql.functions import *

In [0]:
df_bronze_diagnosis = spark.readStream.table('healthcare.bronze.diagnosis')
silver_table = 'healthcare.silver.diagnosis'
checkpoint_path = '/Volumes/healthcare/silver/checkpoint/silver_diagnosis_files/v1' 

In [0]:
df_clean_diagnosis = df_bronze_diagnosis.dropDuplicates(['diagnosis_code']).withColumn('Updated_at', current_timestamp())

In [0]:
def merge_diagnosis_data(batch_df,batch_id):
    if not spark.catalog.tableExists(silver_table):
        batch_df.write.format('delta').mode('append').saveAsTable(silver_table)
        return
    
    from delta.tables import DeltaTable

    final_table = DeltaTable.forName(spark,silver_table)

    (
        final_table.alias('target')
        .merge(
            batch_df.alias('source'),
            'target.diagnosis_code = source.diagnosis_code'
         )
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute()
    )

(
    df_clean_diagnosis
        .writeStream
        .foreachBatch(merge_diagnosis_data)
        .option('checkpointLocation',checkpoint_path)
        .trigger(availableNow = True)
        .start()
)

